# 🏗️ Synthetic Floor 7-Stage — GPU Blender Renderer (Colab)

This notebook renders a **realistic 22×14m construction room** across 7 progressive stages using Blender Cycles + GPU (OptiX/CUDA) on Google Colab.

### What it produces
- **30-60 second smartphone-style video** per stage
- **RGB frames** (PNG, photorealistic interior)
- **Depth maps** (EXR, metric meters)
- **Segmentation masks** (Object Index per category)
- **Camera path JSON** (intrinsics + per-frame cam-to-world)
- **IFC4 BIM files** + OBJ/GLB/PLY meshes

### The 7 Construction Stages
| Stage | Description |
|-------|-------------|
| 1 | Bare RC frame — columns, slab, ceiling, pipes, masonry sills. Perimeters open. |
| 2 | Back (north) wall cinderblock — 2 window openings formed |
| 3 | Left (west) wall partial masonry — door rough opening |
| 4 | Full perimeter enclosure — all walls raw cinderblock |
| 5 | Partial rendering — lower half of left wall plastered white |
| 6 | Full left wall plaster + grey metal door installed |
| 7 | Fully finished — all walls painted, glazed windows, ceiling + 6 recessed lights |

### Why it looks realistic (not black!)
- Window openings are **real gaps** in the wall geometry
- **Light Portals** at every opening guide Cycles sampling
- **12 bounce** light paths (diffuse=8, glossy=4, transmission=12)
- **6 ceiling Area Lights** (80W warm white) for stage 7
- **Extra portals** at open perimeters for stages 1-3
- **Exposure=1.2** + Filmic for natural brightness
- **OpenImageDenoise** for clean output

---

**⚠️ Before running:** Go to *Runtime → Change runtime type* → select **T4 GPU** (free) or **A100** (paid).

---
## 1. Verify GPU + Clone Repository

In [ ]:
import subprocess, os, sys

# Verify GPU is available
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '-L']).decode().strip()
    print(f'✅ GPU detected: {gpu_info}')
except FileNotFoundError:
    print('❌ No GPU! Go to Runtime → Change runtime type → T4/A100')
    sys.exit(1)

# Clone the repository
if not os.path.isdir('/content/MyCon'):
    !git clone --depth 1 https://github.com/sayyedalimrj/MyCon.git /content/MyCon
    print('✅ Repository cloned.')
else:
    print('✅ Repository already present.')

%cd /content/MyCon/repository
print(f'Working directory: {os.getcwd()}')

---
## 2. Install Blender 4.2 LTS (Portable Build)

Downloads the official Blender 4.2 portable Linux build (~200MB), extracts it, and verifies Cycles can detect the GPU.

In [ ]:
%%bash
set -e

BLENDER_VERSION="4.2.3"
BLENDER_TAR="blender-${BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL="https://download.blender.org/release/Blender4.2/${BLENDER_TAR}"

if [ ! -f "/content/blender/blender" ]; then
  echo "⏳ Downloading Blender ${BLENDER_VERSION}..."
  wget -q "${BLENDER_URL}" -O /tmp/blender.tar.xz
  mkdir -p /content/blender
  tar -xf /tmp/blender.tar.xz -C /content/blender --strip-components=1
  rm /tmp/blender.tar.xz
  echo "✅ Blender installed."
else
  echo "✅ Blender already installed."
fi

# Install runtime libs Blender needs
apt-get -qq install -y libxrender1 libxi6 libxkbcommon0 libsm6 libgl1 > /dev/null 2>&1

# Verify version and GPU detection
echo ""
echo "Blender version:"
/content/blender/blender --version 2>&1 | head -2
echo ""
echo "GPU devices visible to Cycles:"
/content/blender/blender -b --python-expr "
import bpy
prefs = bpy.context.preferences.addons['cycles'].preferences
for backend in ['OPTIX', 'CUDA']:
    try:
        prefs.compute_device_type = backend
        prefs.get_devices()
        for d in prefs.devices:
            if d.type != 'CPU':
                print(f'  ✅ {d.name} ({d.type})')
    except:
        pass
" 2>&1 | grep '✅'

---
## 3. Install Python Dependencies

These run on the **host** Python (Colab kernel). Blender uses its own internal Python — no conflict.

In [ ]:
!pip install -q numpy Pillow PyYAML trimesh imageio imageio-ffmpeg ifcopenshell scipy
print('✅ All Python dependencies installed.')

---
## 4. Quick Smoke Test — Stage 7, Debug Preset (~30-60s)

Renders 30 frames at 480×270 with 32 samples. This confirms:
- GPU detection works
- Mesh imports correctly
- Materials assign properly
- Light portals + ceiling lights are placed
- Compositor outputs RGB + Depth + Seg
- Camera path animates

If this cell succeeds, the full render will work too.

In [ ]:
!PYTHONPATH=examples/synthetic_floor_7stage/src \
    python3 examples/synthetic_floor_7stage/scripts/run_blender_gpu.py \
        --blender /content/blender/blender \
        --preset debug \
        --stages 7

---
## 5. Preview the Smoke Test Output

If you see a **bright, well-lit interior** (not black!), the lighting system works correctly.

In [ ]:
from PIL import Image
from pathlib import Path
import numpy as np
from IPython.display import display

rgb_dir = Path('examples/synthetic_floor_7stage/output/blender_renders/stage_07/rgb')
frames = sorted(rgb_dir.glob('frame_*.png'))
print(f'🎬 {len(frames)} RGB frames rendered')

if frames:
    # Show first, middle, and last frames
    for idx, label in [(0, 'First'), (len(frames)//2, 'Middle'), (-1, 'Last')]:
        img = Image.open(frames[idx])
        print(f'\n{label} frame ({frames[idx].name}): {img.size[0]}x{img.size[1]}')
        # Check if the image is mostly black (brightness test)
        arr = np.array(img)
        mean_brightness = arr.mean() / 255.0
        print(f'  Mean brightness: {mean_brightness:.3f} (should be >0.1 for a lit scene)')
        display(img.resize((640, 360)))
else:
    print('❌ No frames found! Check the Blender render log above.')

In [ ]:
# Preview depth map
import matplotlib.pyplot as plt

depth_dir = Path('examples/synthetic_floor_7stage/output/blender_renders/stage_07/depth')
exrs = sorted(depth_dir.glob('frame_*.exr'))
if exrs:
    try:
        import imageio.v3 as iio
        arr = np.array(iio.imread(exrs[len(exrs)//2]))
        if arr.ndim == 3:
            arr = arr[..., 0]  # take first channel
        finite = arr[np.isfinite(arr) & (arr < 1e4)]
        if finite.size:
            print(f'📏 Depth range: {finite.min():.2f} – {finite.max():.2f} meters')
            fig, ax = plt.subplots(1, 1, figsize=(10, 5))
            im = ax.imshow(np.where(arr < 1e4, arr, np.nan), cmap='turbo')
            plt.colorbar(im, label='Distance (m)')
            ax.set_title('Depth Map (middle frame)')
            ax.axis('off')
            plt.tight_layout()
            plt.show()
    except ImportError:
        print('⚠️ imageio not available for EXR preview')
else:
    print('No depth EXR files found.')

---
## 6. Full 7-Stage Render — Balanced Preset

| Setting | Value |
|---------|-------|
| Resolution | 960×540 |
| Samples | 96 |
| Frames/stage | 120 (4s @ 30fps) |
| Motion blur | Yes |
| Total video | 7 × 4s = 28 seconds |
| Est. time (T4) | ~15-25 min |
| Est. time (A100) | ~5-10 min |

For a **30-60 second video per stage**, use `--frames 900` or `--frames 1800`.

In [ ]:
# Change --frames to 900 for 30s video, or 1800 for 60s video
# Change --preset to 'hq' for 1280x720 @ 192 samples (slower but prettier)

!PYTHONPATH=examples/synthetic_floor_7stage/src \
    python3 examples/synthetic_floor_7stage/scripts/run_blender_gpu.py \
        --blender /content/blender/blender \
        --preset balanced \
        --frames 120 \
        --device OPTIX

---
## 7. High Quality — Single Stage with 30s Video

Render one stage at full quality for paper/thesis figures.

In [ ]:
# Choose which stage to render in HQ
STAGE = 7  # Change to 1-7

!PYTHONPATH=examples/synthetic_floor_7stage/src \
    python3 examples/synthetic_floor_7stage/scripts/run_blender_gpu.py \
        --blender /content/blender/blender \
        --preset hq \
        --stages {STAGE} \
        --frames 900 \
        --device OPTIX

---
## 8. Preview All 7 Stages Side by Side

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

stage_names = [
    '1: RC Frame + Sills',
    '2: Back Wall Framing',
    '3: Left Wall Partial',
    '4: Full Enclosure',
    '5: Partial Plaster',
    '6: Left Plaster + Door',
    '7: Fully Finished',
]

for i in range(7):
    sid = i + 1
    rgb_dir = Path(f'examples/synthetic_floor_7stage/output/blender_renders/stage_{sid:02d}/rgb')
    frames = sorted(rgb_dir.glob('frame_*.png'))
    if frames:
        img = Image.open(frames[len(frames)//2])
        axes[i].imshow(img)
        axes[i].set_title(stage_names[i], fontsize=11)
    else:
        axes[i].text(0.5, 0.5, 'Not rendered', ha='center', va='center')
        axes[i].set_title(stage_names[i], fontsize=11)
    axes[i].axis('off')

axes[7].axis('off')  # hide 8th subplot
plt.suptitle('Construction Progress — 7 Stages', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('examples/synthetic_floor_7stage/output/7_stages_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved overview to output/7_stages_overview.png')

---
## 9. Encode MP4 Videos + Download

The orchestrator already encodes MP4s automatically. This cell lists what's available and lets you download them.

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display

video_dir = Path('examples/synthetic_floor_7stage/output/video')
mp4s = sorted(video_dir.glob('*.mp4'))
print(f'📹 {len(mp4s)} video file(s) available:\n')
for mp4 in mp4s:
    size_mb = mp4.stat().st_size / 1024 / 1024
    print(f'  {mp4.name:30s} ({size_mb:.1f} MB)')
    display(FileLink(str(mp4)))

---
## 10. Inspect Manifest + Camera Path

In [ ]:
import json
from pathlib import Path

manifest_path = Path('examples/synthetic_floor_7stage/output/manifests/manifest_blender_gpu.json')
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print('📋 Manifest summary:')
    print(f'  Project: {manifest.get("project", "?")}')
    print(f'  Stages rendered: {list(manifest.get("stage_files", {}).keys())}')
    for sid, files in manifest.get('stage_files', {}).items():
        n_files = sum(1 for v in files.values() if isinstance(v, dict) and v.get('exists'))
        print(f'  Stage {sid}: {n_files} artefact files')
else:
    print('⚠️ Manifest not found — run the full render first.')

# Camera path
cam_path = Path('examples/synthetic_floor_7stage/output/blender_renders/stage_07/camera_path.json')
if cam_path.exists():
    cam = json.loads(cam_path.read_text())
    print(f'\n📷 Camera path (stage 7):')
    print(f'  Schema: {cam["schema_version"]}')
    print(f'  Frames: {len(cam["frames"])}')
    print(f'  FPS: {cam["fps"]}')
    intr = cam['intrinsics']
    print(f'  Resolution: {intr["width_px"]}x{intr["height_px"]}')
    print(f'  Lens: {intr["lens_mm"]:.2f}mm (FOV: {intr["horizontal_fov_deg"]:.1f}°)')

---
## 11. Download Everything as ZIP

In [ ]:
import shutil
from IPython.display import FileLink

output_dir = 'examples/synthetic_floor_7stage/output'
zip_path = '/content/synthetic_floor_7stage_output'

print('📦 Compressing output folder...')
shutil.make_archive(zip_path, 'zip', output_dir)
zip_file = f'{zip_path}.zip'
size_mb = Path(zip_file).stat().st_size / 1024 / 1024
print(f'✅ ZIP ready: {size_mb:.1f} MB')
display(FileLink(zip_file))

---
## Notes

### Quality Presets

| Preset | Resolution | Samples | Frames | Motion Blur | Use Case |
|--------|-----------|---------|--------|-------------|----------|
| `debug` | 480×270 | 32 | 30 | No | Smoke test (30s) |
| `balanced` | 960×540 | 96 | 120 | Yes | Development (15-25 min) |
| `hq` | 1280×720 | 192 | 180 | Yes | Final figures (30+ min) |

### Custom Render

```bash
# 60-second video at full HD
--preset hq --frames 1800 --resolution 1920 1080 --samples 256

# Just stages 1 and 7 for comparison
--preset balanced --stages 1 7 --frames 900

# Resume after a crash (skip already-complete stages)
--preset balanced --resume
```

### Why Early Stages Are Bright (Not Dark)

Stages 1-3 have **open perimeters** (no walls on some sides) — the sky is directly visible and floods the scene with ambient light. The renderer also places extra light portals at the open gaps to help Cycles find the sky efficiently.

### Why Stage 7 Is Bright (Not Dark)

Stage 7 is a **fully enclosed room** (walls + ceiling). It's lit by:
1. **3 large windows** (2.4m × 1.8m) with light portals → daylight floods in
2. **6 ceiling Area Lights** (80W warm white) → simulates recessed LED panels
3. **12 bounce counts** → light reflects realistically off white walls
4. **Exposure=1.2** → compensates for the indoor brightness loss